# 🏔️ ALTAY LLM — TEK TIKLA ÇALIŞTIR

**Ne yapar:** Qwen 2.5 7B'yi Türkçe + İngilizce için özelleştirir
**Süre:** ~2 saat

## ▶️ NASIL ÇALIŞTIRILIR

1. **Çalışma Zamanı → Tümünü çalıştır (Ctrl+F9)**
2. Aşağıda token kutucuğu çıkacak → token'ı yapıştır → Enter
3. 2 saat bekle — bitince modelin hazır! 🎉

> ⚠️ **İLK ÇALIŞTIRMADA** "GPU'a bağlanın" diye bir uyarı çıkabilir, "T4 GPU" veya "Connect" butonuna tıkla. Sonra tekrar Ctrl+F9 yap.


In [ ]:
# ============================================================
# KURULUM
# ============================================================
import torch
print(f"GPU: {torch.cuda.get_device_name() if torch.cuda.is_available() else 'YOK - Lütfen üstteki GPU uyarısına tıkla!'}")
!pip install -q unsloth xformers trl accelerate bitsandbytes datasets

import os
from getpass import getpass
from huggingface_hub import login, HfApi, create_repo

HF_TOKEN = getpass('Hugging Face token\'ını yapıştır: ')
login(token=HF_TOKEN)
print("Hazır!")


In [ ]:
# ============================================================
# MODELİ YÜKLE
# ============================================================
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-7B",
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
)

t = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f"Model hazır! {t:.1f}M parametre eğitilecek")


In [ ]:
# ============================================================
# VERİYİ HAZIRLA
# ============================================================
from datasets import load_dataset
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

oa = load_dataset("OpenAssistant/oasst1", split="train", streaming=True)

threads = {}
for item in oa:
    tid = item["message_tree_id"]
    if tid not in threads:
        threads[tid] = []
    threads[tid].append(item)
    if len(threads) >= 6000:
        break

SISTEM = "Sen, Altay LLM'sin. Türkçe ve İngilizce her konuda yardımcı olan bir asistansın."

konusmalar = []
for tid, msgs in threads.items():
    msgs.sort(key=lambda x: x["created_date"])
    dil = msgs[0].get("lang", "en")
    if dil not in ["tr", "en"]:
        continue
    if dil == "en" and len(konusmalar) > 500:
        continue
    conv = [{"role": "system", "content": SISTEM}]
    for m in msgs:
        conv.append({"role": "assistant" if m["role"] == "assistant" else "user", "content": m["text"]})
    if len(conv) >= 2:
        konusmalar.append(tokenizer.apply_chat_template(conv, tokenize=False))

split = int(len(konusmalar) * 0.95)
train_ds = Dataset.from_dict({"text": konusmalar[:split]})
eval_ds = Dataset.from_dict({"text": konusmalar[split:]})
print(f"{len(train_ds)} egitim, {len(eval_ds)} eval")


In [ ]:
# ============================================================
# EĞİTİM (~2 SAAT)
# ============================================================
print("EĞİTİM BAŞLIYOR! Colab'i kapatma.")

Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="./cikti", per_device_train_batch_size=2,
        gradient_accumulation_steps=4, learning_rate=2e-4,
        warmup_ratio=0.05, lr_scheduler_type="cosine",
        logging_steps=10, eval_steps=50, save_steps=200,
        evaluation_strategy="steps", save_strategy="steps",
        save_total_limit=2, max_grad_norm=0.3,
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        gradient_checkpointing=True, report_to="none",
    ),
    train_dataset=train_ds, eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8),
).train()

print("EĞİTİM TAMAM!")


In [ ]:
# ============================================================
# MODELİ KAYDET
# ============================================================
model.save_pretrained_merged("./altay-son", tokenizer, save_method="merged_16bit")

api = HfApi(token=HF_TOKEN)
create_repo("cntzcn10/altay-llm", exist_ok=True)
api.upload_folder(repo_id="cntzcn10/altay-llm", folder_path="./altay-son", path_in_repo=".")
print("Model yayında: https://huggingface.co/cntzcn10/altay-llm")


In [ ]:
# ============================================================
# TEST
# ============================================================
model, tokenizer = FastLanguageModel.from_pretrained(
    "./altay-son", max_seq_length=4096,
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

for soru in ["Merhaba, kimsin?", "Türkiye'nin başkenti neresi?"]:
    msg = [{"role": "system", "content": SISTEM}, {"role": "user", "content": soru}]
    girdi = tokenizer.apply_chat_template(msg, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    cikti = model.generate(girdi, max_new_tokens=100)
    print(f"\n🧑 {soru}")
    print(f"🤖 {tokenizer.decode(cikti[0][girdi.shape[1]:], skip_special_tokens=True)}")

print("\n🎉 ALTAY LLM HAZIR!")
